In [1]:
from langchain.chat_models import init_chat_model
from langchain_community.chat_models import ChatTongyi
from langchain_openai import ChatOpenAI
from config import settings

model = ChatOpenAI(
    model=settings.openai_model_name,
    api_key=settings.openai_api_key,
    base_url=settings.openai_base_url
)

In [2]:
# 调用glm大模型接口进行简答的问答

message = "新中国成立的时间是什么时候？"

response = model.invoke(message)

print(response)



content='新中国成立的时间是1949年10月1日。当天，毛泽东主席在北京天安门广场宣告中华人民共和国成立，并亲自升起了第一面五星红旗。这一天被定为中国的国庆节。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 10, 'total_tokens': 52, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'glm-4-flash', 'system_fingerprint': None, 'id': '20260602170536d13019b543464578', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--9bec88aa-503d-4b5a-b564-767c4d58d1ab-0' usage_metadata={'input_tokens': 10, 'output_tokens': 42, 'total_tokens': 52, 'input_token_details': {}, 'output_token_details': {}}


In [3]:
"""
模型结构化输出
1. 使用with_structured_output 函数
2. 设置系统提示词让模型输出json格式
"""

from typing import Optional, List
from pydantic import BaseModel, Field


class UserInfo(BaseModel):
    """提取有关用户的信息。"""
    name: str = Field(description="用户的姓名")
    age: Optional[int] = Field(description="用户的年龄")
    occupation: str = Field(description="用户的职业")
    skills: List[str] = Field(default_factory=list, description="用户的技能列表")

llm_struct_output = model.with_structured_output(UserInfo, method="function_calling")
message = "我的名字是张三，今年 28 岁。我在一家科技公司担任后端工程师，熟悉 Python, Go 和 Kubernetes。"
response = llm_struct_output.invoke(message)
print(response)

name='张三' age=28 occupation='后端工程师' skills=['Python', 'Go', 'Kubernetes']


In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

system_message = '''
你是一个关键信息提取助手，请根据用户的对话提取一下关键信息：
1. name: 用户姓名
2. age: 用户的年龄
3. occupation: 用户的职业
4. skills: 用户的技能列表
以严格的json格式返回最终的数据
如下给出一个返回的参考例子：
{
    name: '小明',
    age: 21,
    occupation: '前端',
    skills: ['Vue', 'React']
}
'''
message = "我的名字是张三，今年 28 岁。我在一家科技公司担任后端工程师，熟悉 Python, Go 和 Kubernetes。"
response = model.invoke([SystemMessage(system_message),HumanMessage(message)])
print(response.content)

{
    "name": "张三",
    "age": 28,
    "occupation": "后端工程师",
    "skills": ["Python", "Go", "Kubernetes"]
}


In [5]:
from langchain_core.messages import ToolMessage
import datetime
from langchain.tools import tool


@tool
def get_current_time():
    """获取今天的日期。当你需要知道当前日期或时间相关信息时很有用。"""
    return datetime.datetime.now().strftime("%Y-%m-%d")

llm_with_tools = model.bind_tools(tools=[get_current_time])
message = "今天是几月几号？"
response = llm_with_tools.invoke(message)
print(response)

tool_results = []
for tool_call in response.tool_calls:
    result = get_current_time.invoke(tool_call["args"])
    tool_results.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
print(tool_results)

final_response = llm_with_tools.invoke([HumanMessage(content=message), response] + tool_results)
print(final_response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 135, 'total_tokens': 141, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'glm-4-flash', 'system_fingerprint': None, 'id': '202606021705444c677595995040df', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--c9c6355c-eaff-414a-917d-2674085cabd6-0' tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'call_-7566472335226567941', 'type': 'tool_call'}] usage_metadata={'input_tokens': 135, 'output_tokens': 6, 'total_tokens': 141, 'input_token_details': {}, 'output_token_details': {}}
[ToolMessage(content='2026-06-02', tool_call_id='call_-7566472335226567941')]
content='今天是2026年6月2日。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 149, 'total_tokens': 160, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_